### How weakest link labels are created
The weakest link is not directly given in the raw data. Instead it is derived.

For each sample
* We look at all indicator columns (for example **ForwardHead**, **LeftArmFallForward**, etc)
* We find the indicator with the maximum deviation value
*The name of that indicator becomes the **WeakestLink** label

This logic is implemented with 
`scores["WeakestLink"]=scores[indicators_cols].idmax(axis=1)`

### What is **HasTie** and why dies it exist?
Sometimes two or more indicators have exactly the same maximum value for a sample.
In that case, the “true” weakest link is ambiguous.

The **HasTie** flag is used to mark these cases:

`scores["HasTie"] = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)
`
* HasTie = True means the label is ambiguous.

* We track this explicitly so we can later analyze or exclude these samples if needed.

This step is important because **machine learning models assume that labels are correct and unambiguous.**

## Why Some Weakest Link Classes Are Removed
After creating labels, we analyze class frequencies.

Some weakest link classes appear only:
* once, or
* a handful of times

These very small classes cause multiple problems:
* Stratified cross-validation breaks or becomes unstable.
* Per-class F1 scores collapse to zero
* Models overfit noise instead of learning general patterns.

Therefore, some weakest link classes are explicitly removed before training, for example:

`DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift"
}
`

This is not “cheating” — it is a standard data-quality step to ensure that the task is learnable with the available data.

## Why Feature Scaling Is Used (StandardScaler)

Many models assume that input features are on comparable scales.

In our data:
* some deviation values may be large,
* others may be small,
* and the units are not directly comparable.

**StandardScaler** transforms each feature independently so that:
* the mean becomes approximately 0
* the standard deviation becomes approximately 1

This does not mean all values become 0 or 1.
It means:
* values are expressed in terms of “how far from the average” they are.

This is crucial for:
* Logistic Regression
* k-Nearest Neighbors
* Linear Discriminant Analysis

Without scaling, these models can behave unpredictably.

## Why Accuracy Alone Is Not Enough

The dataset is class-imbalanced:

* some weakest links appear very often,
* others appear rarely.

A model can get high accuracy by:

* always predicting a common class,
* while completely failing on rare classes.

Therefore, we evaluate models using:

* **Macro F1 score** (treats all classes equally),
* **Weighted F1 score** (accounts for class frequency),
* Precision and recall (macro-averaged).

Macro F1 is especially important because:

It tells us whether the model performs reasonably across all weakest links, not just the most frequent ones.

## Why Stratified Cross-Validation Is Used

Instead of training once, we use cross-validation:

* the data is split into multiple folds,
* each fold is used once as validation,
* results are averaged.

Stratified cross-validation ensures that:

* each fold contains roughly the same class proportions as the full dataset.

This prevents evaluation results from depending on a “lucky” or “unlucky” split.

After removing rare classes, we are able to use 5-fold stratified cross-validation, which gives more stable and reliable estimates.

## Multiple Models Are Compared

We compare:

* Logistic Regression (linear, interpretable),
* Gaussian Naive Bayes (probabilistic, simple),
* k-Nearest Neighbors (distance-based),
* Linear Discriminant Analysis (distribution-based),
* XGBoost and LightGBM (non-linear boosting models),
* plus a dummy baseline.

All models are:

* trained on the same data,
* evaluated using the same metrics,
* compared using the same cross-validation splits.

This makes comparisons fair and meaningful.

## Why Logistic Regression Regularization Is Tuned

Logistic Regression uses regularization to prevent overfitting.

The parameter C controls this:

* **low C** → strong regularization → simpler model
* **high C** → weak regularization → more flexible model

After removing noisy and rare classes:

* the dataset becomes cleaner,
* stronger regularization becomes unnecessary,
* higher C performs better.

This shows that data cleaning and model tuning are connected.

## Why Boosting Models Perform Best
Boosting models (XGBoost, LightGBM):

* combine many small decision trees,
* model non-linear relationships,
* handle feature interactions automatically.

Once data quality and evaluation are solid, these models:

* outperform linear models,
* achieve higher macro F1 scores,
* better balance performance across classes.

XGBoost consistently emerges as the strongest overall model in this setup.

--

## More indepth explenation

### Common pipeline components

Most models were implemented using a machine learning pipeline. A pipeline ensures that all preprocessing steps are applied consistenlty and correctly during both training and cross-validation

#### StandardScaler

StandardScaler transforms each feature so that:
* the mean becomes 0
* the standard deviation becomes 1

Why necessary?

The 38 input features (angle deviations and NASM deviations) exist on different numeric scales. Modesl that rely on distances or weight magnituted are sensitive to feature scale. Without scaling, features with larger numeric ranges would dominate the learning process.

Effect on performance:
Improves convergence and stability for Logistic Regression. Is essential for K-Nearest Neighbors, which relies directly on distance calculations. Often improves Linear Discriminant Analysis. DOes not harm Gaussian Naive Bayes, even though it is not strictly required.

### Dummy Classifier (Baseline)

This model always predicts the most common class in the training data

#### Purpose
It serves as a baseline. Any real model myst perform better than this to be considered useful.

#### Observed behavior
Accyracy may appear moderate in imbalanced datasets. Macro F1-score is extremely low because minority classes are never predicted.

#### Why important?
It demonstrates that higher accuracy alone is misleading in imbalanced multiclass problems and justifies the use of macro F1-score

### Logisit Regression (Multiclass)

Logistic Regression models the probability of each class using a linear decision boundary.

#### Reqularization Parameter C
##### Def:
C is the inverse of regularization (technique in machine learning to prevent overfitting) strenght.
* Smaller C -> stronger regularization
* Larger C -> weaker regularization

#### What regularization means
Regularization penalizes large model weights, reducing overfitting and improving generalization to unseen data.

C = 1.0 is the standard default and serves as a reasonable baseline
C = 0.3 applies stronger regularization

#### Impact on results:
Stronger regularization (C=0.3) reduces sensitivity to noise. Produces more stable decision boundaries. Often improves macro F1-score in imbalanced datasets. Can slightly reduce accuracy but improves fairness across classes.

### Class Weight = "balanced"

#### Def
Class weights are automatically adjusted to be inversely proportional to class frequency.

#### Why necessary?
The dataset contains sever class imbalance, with some classes having very few samples. Without weighting, the model would prioritize majority classes.

#### Effect?
Improves recall for minority classes. Increases macro F1-score. May slightly reduce accuracy, which acceptable given the project goal.

### Solver = "lbfgs"

#### Def
This is an optimization algorithm used to find the best model parameters. 

#### Why chosen:
Works well for multinomial logistic regression. Stable and efficient for medium-sized datasets

### Max iteration = 5000
Maximum number of optimization steps during training

#### Why increased/this high?
Small classes and strong regularization can slow convergence. Increasing the iteration limit prevents premature stopping.

### Gaussian Naive Bayes

Gaussian Naive Bayes assumes:
* Each feature follows a Gaussian (normal) distribution
* All features are conditionally independent given the class

#### Variance Smoothing
##### Def
A small value added to feature varainces to prevent numerical instability.

#### Why tuned?
If a feature has near-zero varaince in a class, the model can become unstable

#### Effect
* Improves numerical stability
* Excessive smoothing can reduce sensitivity to real differences between classes

#### Because angle and NASM features are correlated, the independence assumption is violated which limits performance

### Linear Discriminant Analysis (LDA)

Linear Discriminant Analysis models each class as a Gaussian distribution with:
* Different means
* A shared covariance matrix
This allows it to exploit correlations between features

What your per-class table actually tells you

Let’s interpret the most important rows:

🔴 Extremely small classes
LeftHeelRises: full=2, train=2, test=0
LeftKneeMovesInward: full=3, train=2, test=1


Facts:

These classes have 2 training samples total

One of them has zero test samples

F1 score is necessarily 0 or undefined

📌 No model can learn a reliable decision boundary from 2 samples
This is a data limitation, not a modeling mistake.

🟠 Small but learnable classes
RightAsymmetricalWeightShift: train=15 → F1=0.40
LeftAsymmetricalWeightShift: train=58 → F1=0.48


These classes:

Exist in training and test

Have moderate imbalance

Show some learning signal

These can benefit from imbalance-aware training.

🟢 Large, well-learned classes
LeftArmFallForward: train=504 → F1=0.66
RightKneeMovesOutward: train=222 → F1=0.91


These dominate:

Weighted F1

Overall accuracy

Per-class F1 analysis revealed substantial class imbalance. Several classes contained fewer than five training samples, making reliable learning impossible regardless of model choice. While class weighting and tree-based models improved performance for moderately rare classes, extremely small classes remained unlearnable. This limitation reflects data availability rather than model deficiency

### Effect of Removing Low-Quality Classes on Logistic Regression Regularization

Logistic regression uses a regularization mechanism to control model complexity and prevent overfitting. This mechanism penalizes large model coefficients, thereby constraining how strongly individual features can influence predictions. The strength of this penalty is controlled by the parameter C, which represents the inverse of regularization strength. A larger value of C corresponds to weaker regularization, allowing the model to fit the data more flexibly, while a smaller value of C corresponds to stronger regularization, forcing the model to remain more constrained.

After removing the four weakest-link classes LeftHeelRises, RightHeelRises, LeftKneeMovesInward, and RightAsymmetricalWeightShift, the performance behavior of logistic regression changed in a consistent and explainable way. Specifically, logistic regression with a higher value of C performed better, while logistic regression with a lower value of C performed worse.

This effect can be explained by changes in label quality, class imbalance, and noise levels in the dataset.

The removed classes had extremely low sample counts and consistently poor predictive performance, with F1 scores close to zero. Such classes introduce label noise, meaning that the relationship between input features and output labels is weak, inconsistent, or ambiguous. In the presence of label noise, stronger regularization can be beneficial because it prevents the model from fitting unreliable patterns. However, once these noisy and low-support classes were removed, the remaining dataset became cleaner and more internally consistent.

With fewer noisy classes, the model no longer needed strong regularization to suppress unstable patterns. Instead, allowing greater flexibility became advantageous. A higher value of C enabled logistic regression to assign more meaningful weights to the 38 movement predictors, capturing subtler but real distinctions between the remaining weakest-link classes. As a result, precision and macro-averaged F1 score improved.

In contrast, a lower value of C imposed overly strong regularization on the cleaned dataset. This excessive constraint limited the model’s ability to exploit informative features, leading to underfitting. Underfitting occurs when a model is too simple to capture important structure in the data, resulting in poorer class separation and reduced macro-averaged performance metrics.

Additionally, the use of balanced class weighting further reduced the need for strong regularization. Class weighting compensates for remaining class imbalance by adjusting the importance of minority classes during training. Since imbalance was already being handled explicitly, strong regularization was no longer necessary as a corrective mechanism.

In summary, removing the four weakest-link classes reduced label noise and improved class reliability. In this cleaner learning environment, weaker regularization (higher C) allowed logistic regression to better utilize informative predictors, while stronger regularization (lower C) unnecessarily constrained the model, leading to inferior performance. This explains why logistic regression with a higher value of C performed better after class filtering, whereas logistic regression with a lower value of C performed worse.